# Add-on experiments — do more signals actually help?

A running log of "what if we bolt one more idea onto the champion?" experiments. Three
directions tested here (the user's three ideas):

1. **GMM soft clustering** — let each asset belong *partially* to several families.
2. **Weighted history** — use older data too, down-weighted (recency half-life), instead of a flat 120-day window.
3. **RSI / MACD** — add a classic technical indicator as an extra sleeve.

**How we judge (the house rule):** not raw PnL and not just the full-window Score, but the
**H1/H2 split** — the weak first half vs the strong second half of the 250-day window. The
champion's whole reason to exist is that it is *regime-balanced* (H1 ≈ H2). A change that lifts
the full score by hurting H1 is a **bad** change here.

**The bar to beat:** `family_cluster_ownrevert` — **full 304.3, H1 295.9, H2 312.6, Sharpe 2.88.**

> TL;DR of everything below: **none of the three beats the champion.** Each fails in an
> instructive way that reinforces the same lesson — the engine wants a *sharp, recent, hard*
> grouping and a *fast* reversion signal; smoothing it, feeding it older data, or piling on more
> reversion all fight its nature. Details + numbers per section.

In [1]:
# The reusable engine lives in addons_lab.py (rebuilds the champion as a configurable closure).
# make_strategy(family_mode=..., ew_halflife=..., rsi_budget=...) -> a getMyPosition to score.
# evaluate(name, fn) -> prints full + H1/H2 Score (the judging metric) and returns a dict.
import numpy as np
import pandas as pd
from addons_lab import make_strategy, evaluate, PRC, CFG
from research import featureIC, NUM_TEST_DAYS

print(f"prices: {PRC.shape[0]} instruments x {PRC.shape[1]} days")
results = {}  # collect rows for the summary table at the bottom

prices: 51 instruments x 500 days


## 0. Baseline — the champion (the bar to beat)

In [2]:
results["champion (hard, flat-120)"] = evaluate("champion (hard, flat-120)", make_strategy("hard"))

  champion (hard, flat-120)          full  304.3 (Sh 2.88) | H1  295.9 | H2  312.6 | turn $373,544


## Idea 3 — RSI / MACD signals

Two questions before building anything: **(a) is the signal even predictive?** and
**(b) is it NEW, or just a re-derivation of the own-price sleeve we already trade?**

**Caveat on `featureIC`:** it measures *cross-sectional* rank power (which asset beats the
others tomorrow). But our own-price edge is *time-series* (when to be long/short each asset),
which IC under-measures — that's why the own-z signal shows a tiny cross-sectional IC yet
backtests to 304. So IC is a **screening hint here, not the verdict** — the backtest is.

In [3]:
# candidate signals (higher = expected to outperform)
def feat_ownz(p, w=3):
    logP = np.log(p); prior = logP[:, -w-1:-1]
    z = (logP[:, -1] - prior.mean(1)) / (prior.std(1) + 1e-12)
    f = -z; return f - f.mean()

def feat_rsi(p, n=14, revert=True):
    d = np.diff(p[:, -(n+1):], axis=1)
    gain = np.clip(d, 0, None).mean(1); loss = np.clip(-d, 0, None).mean(1)
    rsi = 100 - 100/(1 + gain/(loss + 1e-12))
    f = (50 - rsi) if revert else (rsi - 50); return f - f.mean()

def _ema_last(x, span):
    a = 2/(span+1); w = (1-a) ** np.arange(x.shape[1]-1, -1, -1)
    return (x*w).sum(1)/w.sum()

def feat_macd(p, fast=12, slow=26, sig=9, revert=False):
    logP = np.log(p); look = slow+sig+5; seg = logP[:, -look:]
    ms = np.zeros((p.shape[0], seg.shape[1]))
    for k in range(1, seg.shape[1]):
        ms[:, k] = _ema_last(seg[:, :k+1], fast) - _ema_last(seg[:, :k+1], slow)
    hist = ms[:, -1] - _ema_last(ms, sig)
    f = -hist if revert else hist; return f - f.mean()

print("next-day IC (walk-forward). |IC|>~0.03 & |t|>2 = real; sign +=as-written predicts up.")
for nm, fn in [("own-z w=3 (baseline sleeve)", lambda p: feat_ownz(p,3)),
               ("RSI n=5  (reversion)", lambda p: feat_rsi(p,5)),
               ("RSI n=14 (reversion)", lambda p: feat_rsi(p,14)),
               ("RSI n=20 (reversion)", lambda p: feat_rsi(p,20)),
               ("MACD hist (momentum)", lambda p: feat_macd(p, revert=False))]:
    r = featureIC(fn, PRC, NUM_TEST_DAYS)
    print(f"  {nm:28s} IC={r['meanIC']:+.4f}  t={r['tstat']:+5.2f}")

next-day IC (walk-forward). |IC|>~0.03 & |t|>2 = real; sign +=as-written predicts up.


  own-z w=3 (baseline sleeve)  IC=+0.0062  t=+0.62


  RSI n=5  (reversion)         IC=+0.0012  t=+0.12


  RSI n=14 (reversion)         IC=+0.0174  t=+1.65


  RSI n=20 (reversion)         IC=+0.0164  t=+1.59


  MACD hist (momentum)         IC=-0.0120  t=-1.10


In [4]:
# Redundancy: how much does each signal overlap the own-price sleeve we already trade?
def feature_corr(fA, fB, prc=PRC, ndays=NUM_TEST_DAYS):
    nt = prc.shape[1]; cs = []
    for t in range(nt-ndays, nt):
        a = pd.Series(np.asarray(fA(prc[:, :t]), float))
        b = pd.Series(np.asarray(fB(prc[:, :t]), float))
        dd = pd.DataFrame({"a": a, "b": b}).dropna()
        if dd["a"].nunique() > 1 and dd["b"].nunique() > 1:
            cs.append(dd["a"].corr(dd["b"], method="spearman"))
    return float(np.mean(cs))

base = lambda p: feat_ownz(p, 3)
print("daily signal corr vs own-z (w=3):  ~1 = same bet (adds nothing); ~0 = new info")
for nm, fn in [("RSI n=5", lambda p: feat_rsi(p,5)), ("RSI n=14", lambda p: feat_rsi(p,14)),
               ("MACD hist (mom)", lambda p: feat_macd(p, revert=False))]:
    print(f"  corr(own-z, {nm:16s}) = {feature_corr(base, fn):+.3f}")

daily signal corr vs own-z (w=3):  ~1 = same bet (adds nothing); ~0 = new info


  corr(own-z, RSI n=5         ) = +0.602


  corr(own-z, RSI n=14        ) = +0.332


  corr(own-z, MACD hist (mom) ) = -0.319


**Insight (screen).** Short-window RSI is basically a *copy* of the own-price sleeve
(corr **+0.60** at n=5), so it can't add much. RSI n=14 is the most *distinct* (corr **+0.33**)
and the strongest reversion IC (+0.017) — the only one worth backtesting. MACD-momentum has a
weak *negative* IC: this is a reversion market at every horizon, so momentum loses (matches the
repo's earlier momentum rejection). So we only build the RSI-14 sleeve and size-sweep it.

In [5]:
# The real test: add an RSI-14 reversion sleeve on top of the champion, sweep its budget.
for b in [250_000, 500_000, 1_000_000, 2_000_000]:
    results[f"+ RSI14 {b//1000}k"] = evaluate(f"+ RSI14 budget={b:,}", make_strategy("hard", rsi_budget=b))

  + RSI14 budget=250,000             full  311.1 (Sh 2.90) | H1  268.3 | H2  353.2 | turn $366,982


  + RSI14 budget=500,000             full  299.4 (Sh 2.78) | H1  232.1 | H2  365.5 | turn $358,211


  + RSI14 budget=1,000,000           full  249.3 (Sh 2.39) | H1  157.8 | H2  338.9 | turn $337,500


  + RSI14 budget=2,000,000           full  171.8 (Sh 1.77) | H1   43.5 | H2  305.5 | turn $297,793


**Verdict — Idea 3 (RSI/MACD): not worth it.** A small RSI sleeve nudges the full score up
(+7 at 250k) but does it the *wrong* way — every budget **hurts H1** (296 → 268 → 232 → 43) while
loading more into the already-strong H2. By the H1 rule that's a losing swap. MACD is momentum in
a reversion market — skip. RSI adds no regime-stable edge the own-price sleeve doesn't already have.
See `strategy/family_cluster_rsi.py` for the on-record file.

## Idea 2 — weighted history (EW clustering)

The champion groups families using only the last **120 days** (flat) and ignores older data.
Idea: also use older data but *down-weighted* — a recency **half-life** (today counts 1, each
older day decays). Small half-life = *more* recency-focused than flat-120; large half-life = drags
in the older ~1-year business cycle the idea hoped would help generalisation. We sweep the half-life.

In [6]:
for hl in [20, 40, 60, 80, 120, 200, 320]:
    results[f"ew hl={hl}"] = evaluate(f"ew halflife={hl} (data 350d)", make_strategy("ew", ew_halflife=hl))

  ew halflife=20 (data 350d)         full  307.6 (Sh 2.84) | H1  272.5 | H2  342.5 | turn $372,030


  ew halflife=40 (data 350d)         full  292.5 (Sh 2.81) | H1  300.0 | H2  285.1 | turn $372,325


  ew halflife=60 (data 350d)         full  279.9 (Sh 2.70) | H1  313.9 | H2  245.6 | turn $370,783


  ew halflife=80 (data 350d)         full  302.7 (Sh 2.87) | H1  305.3 | H2  300.2 | turn $370,401


  ew halflife=120 (data 350d)        full  307.4 (Sh 2.93) | H1  277.8 | H2  336.7 | turn $369,765


  ew halflife=200 (data 350d)        full  286.1 (Sh 2.76) | H1  250.0 | H2  321.8 | turn $370,807


  ew halflife=320 (data 350d)        full  293.4 (Sh 2.84) | H1  266.0 | H2  320.6 | turn $370,239


**Verdict — Idea 2 (weighted history): the specific hypothesis is refuted.** No half-life
reliably beats the flat-120 champion. Crucially, the half-lives that use the **most old data**
(200, 320 — the "1-year cycle" idea) are exactly the ones that **drop the weak half** (H1 296 →
250–266). More recency (hl=20) helps H2 but also hurts H1. The most-balanced setting (hl=80) only
*ties* the champion. This re-confirms the repo's standing finding: the family engine wants **fresh,
recent-only** grouping — the day-to-day churn is a feature, not noise to smooth away with history.
On record: `strategy/family_cluster_ewcluster.py`.

## Idea 1 — GMM soft clustering

The champion puts each asset in exactly one family (hard). Idea: let an asset belong *partially*
to several families (soft membership), so its "family move" is a probability-weighted blend.
We PCA the recent returns, fit a Gaussian Mixture in that space, and use the soft memberships to
build each asset's blended family move, then fade its pull-away from that.

In [7]:
results["gmm soft"] = evaluate("gmm soft (6 comp, 8 dims)", make_strategy("gmm"))
# try sharpening: fewer PCA dims pushes memberships toward harder assignments
for dims in [3, 5, 8]:
    evaluate(f"gmm soft (dims={dims})", make_strategy("gmm", GMM_DIMS=dims))

  gmm soft (6 comp, 8 dims)          full  281.1 (Sh 2.74) | H1  279.4 | H2  282.7 | turn $371,504


  gmm soft (dims=3)                  full  311.0 (Sh 2.94) | H1  277.5 | H2  344.1 | turn $371,699


  gmm soft (dims=5)                  full  309.0 (Sh 2.95) | H1  292.3 | H2  325.6 | turn $372,428


  gmm soft (dims=8)                  full  281.1 (Sh 2.74) | H1  279.4 | H2  282.7 | turn $371,504


**Verdict — Idea 1 (GMM soft clustering): balanced but worse.** Soft membership is nicely
regime-balanced (H1 ≈ H2 ≈ 281) but scores **~23 points below** the champion. Blending across
clusters **smooths** each asset's family move, giving a blurrier snap-back — the same reason
"steadier families score worse" showed up before. Sharpening the GMM (fewer dims) only claws back
toward hard clustering, which defeats the purpose. On record: `strategy/family_cluster_gmm.py`.

## Summary + ranking

In [8]:
df = pd.DataFrame(results).T[["full", "h1", "h2", "sharpe", "turn"]].astype(float)
df = df.sort_values("full", ascending=False)
df.insert(3, "h1_vs_champ", (df["h1"] - 295.9).round(1))  # weak-half delta vs champion
print(df.round(1).to_string())
print("\nRule of thumb: only promote a row that beats the champion on H1 (h1_vs_champ > 0) "
      "WITHOUT giving up the full score. Nothing here does.")

                            full     h1     h2  h1_vs_champ  sharpe      turn
+ RSI14 250k               311.1  268.3  353.2        -27.6     2.9  366982.3
ew hl=20                   307.6  272.5  342.5        -23.4     2.8  372030.0
ew hl=120                  307.4  277.8  336.7        -18.1     2.9  369765.0
champion (hard, flat-120)  304.3  295.9  312.6         -0.0     2.9  373543.7
ew hl=80                   302.7  305.3  300.2          9.4     2.9  370401.3
+ RSI14 500k               299.4  232.1  365.5        -63.8     2.8  358211.0
ew hl=320                  293.4  266.0  320.6        -29.9     2.8  370238.9
ew hl=40                   292.5  300.0  285.1          4.1     2.8  372325.3
ew hl=200                  286.1  250.0  321.8        -45.9     2.8  370807.3
gmm soft                   281.1  279.4  282.7        -16.5     2.7  371503.8
ew hl=60                   279.9  313.9  245.6         18.0     2.7  370782.5
+ RSI14 1000k              249.3  157.8  338.9       -138.1     

## What this rules out, and where edge might still live

All three ideas share one failure mode: they try to **capture more of the same reversion**
(smoother families, longer memory, another oscillator) in a market that already reverts, and
the champion is already at that sweet spot. The unexhausted directions are the ones that add a
**genuinely uncorrelated** return stream, not more reversion:

- **Exploit the hidden-rule asymmetries harder**, not the signals — asset 0 has a 10× bigger
  limit and 5× cheaper fees. The champion already boosts it 10×; is that the right number? (Sweep
  `OWN_INST0_MULT` in `tune`-style.)
- **A different alpha family entirely** — e.g. a *lead–lag* bet (does one asset's move predict
  another's tomorrow?) rather than a reversion bet. That would be uncorrelated with both sleeves.
- **Regime *timing* of the two sleeves** — the family sleeve is regime-fragile, the own-price one
  is regime-stable; a cleaner online estimate of "which regime are we in" could re-weight them.

Anything that just adds more reversion (this notebook) has now been shown, three ways, to not help.